In [1]:
import pandas as pd
import os

from dotenv import load_dotenv

load_dotenv()
from groq import Groq

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [18]:
client = Groq()

chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "How do airplanes fly? Be concise."}],
    model="openai/gpt-oss-20b",
    stream=False,
    # reasoning_format="raw",
    include_reasoning=True,
    reasoning_effort="high",
)

In [20]:
print(chat_completion.choices[0].message.content)

Airplanes fly because their wings, shaped like airfoils, generate lift by redirecting airflow downward, creating a pressure drop above the wing. Engines provide forward thrust to overcome drag. When lift exceeds weight and thrust exceeds drag, the aircraft ascends.


# Expected attendance — Groq LLM experiment

Estimates **`expectedAttendance`** for canonical events by sending each event's stored fields to an LLM on Groq.

- Nothing here changes any script or database: `core/canonical_events.db` is opened **read-only**.
- Answers are cached in `notebooks/attendance_cache.json` (per model + prompt version + event data), so re-running a cell never pays twice. Bump `PROMPT_VERSION` after changing the prompt.

Steps: **setup → load & sample Paris events → what the model sees → prompt → call Groq → inspect → compare models**.

In [ ]:
import hashlib
import json
import random
import re
import sqlite3
import time
from datetime import date
from pathlib import Path

import groq
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

# Project root = first folder (here or above) that has core/canonical_events.db,
# so this works whether the kernel starts in notebooks/ or in the project root.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "canonical_events.db").exists()
)
load_dotenv(PROJECT_ROOT / ".env")

CANONICAL_DB = PROJECT_ROOT / "core" / "canonical_events.db"
CACHE_PATH = PROJECT_ROOT / "notebooks" / "attendance_cache.json"

MODEL = "openai/gpt-oss-20b"  # main model for this experiment
COMPARE_MODELS = ["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b"]
PROMPT_VERSION = "v1"  # bump after editing the prompt -> old cached answers are ignored
BATCH_SIZE = 8  # events per request
# Free-tier Groq limits differ per model: qwen allows only 1,000 output tokens
# per minute, and 8 events need ~1,300 -> smaller batches for it.
BATCH_SIZES = {"qwen/qwen3.8-27b": 3}
# Extra request options per model. Qwen on the free tier allows only 1,000
# output tokens/minute and Groq reserves the reply budget up front, so the
# reply is capped at 900 tokens (a batch of 3 uses ~300) -> ~1 request/minute.
MODEL_OPTIONS = {"qwen/qwen3.8-27b": {"reasoning_format": "hidden", "max_completion_tokens": 900}}

client = Groq()
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)
print("project root:", PROJECT_ROOT)

## 1. Load Paris events and pick a test sample

In [ ]:
def load_canonical_events(city):
    """Every canonical event for a city, as its full merged dict (canonical_json)."""
    con = sqlite3.connect(f"file:{CANONICAL_DB}?mode=ro", uri=True)
    rows = con.execute(
        "SELECT id, canonical_key, canonical_json FROM canonical_events WHERE city = ?", (city,)
    ).fetchall()
    con.close()
    events = []
    for row_id, key, raw in rows:
        event = json.loads(raw)
        event["_row_id"] = row_id
        event["_key"] = key
        events.append(event)
    return events


all_events = load_canonical_events("Paris")
print(len(all_events), "Paris events in canonical_events.db")
pd.Series([e.get("source") for e in all_events]).value_counts()

In [ ]:
# How many test events to take from each source (sportsevents24 = all of them).
SAMPLE_PER_SOURCE = {
    "sportsevents24": 14,
    "meetup": 5,
    "stubhub": 5,
    "dice": 4,
    "songkick": 5,
    "bandsintown": 3,
    "eventbrite": 5,
    "openagenda": 4,
}  # Paris has no ticketmaster / universe events in canonical_events.db

rng = random.Random(42)  # fixed seed -> the same sample every run

by_source = {}
for e in all_events:
    by_source.setdefault(e.get("source"), []).append(e)

sample = []
for source, n in SAMPLE_PER_SOURCE.items():
    pool = sorted(by_source.get(source, []), key=lambda e: e["_row_id"])
    picked = []
    # Make sure the sample includes events that carry the source's extra hint.
    if source == "dice":
        picked = rng.sample([e for e in pool if e.get("capacity")], 2)
    elif source == "meetup":
        picked = rng.sample([e for e in pool if e.get("attendees")], 3)
    picked_ids = {e["_row_id"] for e in picked}
    rest = [e for e in pool if e["_row_id"] not in picked_ids]
    picked += rng.sample(rest, min(n - len(picked), len(rest)))
    sample += picked

print(len(sample), "events in the test sample")
pd.DataFrame(
    [
        {"source": e.get("source"), "event": e.get("event_name"), "date": e.get("date"), "venue": e.get("venue_name")}
        for e in sample
    ]
)

## 2. What the model sees for each event

Only fields that say something about crowd size. Coordinates, image/map URLs and internal ids are dropped (they cost tokens and don't help).
The messy source hints are cleaned first:

| Field sent | Comes from | Cleaning |
|---|---|---|
| `venue_capacity` | DICE `capacity` (`"Doors open8:00 pm CEST\n175 capacity"`) | number extracted |
| `rsvp_count` | Meetup `attendees` | dropped when 0 |
| `performer_followers` | StubHub `follow_count` | dropped when 0 |
| `genres` | Songkick `genres` (JSON text) | parsed, dropped when empty |
| `description` | OpenAgenda, Luxembourg | whitespace collapsed, cut to 600 chars |
| `venue_name` | StubHub | dropped when it's actually a price (`"From $114"` - StubHub connector bug) |

In [ ]:
DESCRIPTION_MAX_CHARS = 600

# Copied as-is when present (renamed hint fields are added below).
PLAIN_FIELDS = [
    "event_name", "source", "type", "category", "genre",
    "sport", "tournament", "team_home", "team_away",
    "date", "time", "end_date", "end_time",
    "venue_name", "venue_address", "venue_city", "venue_country",
    "organizer", "description",
    "price_from", "price_min", "price_max", "currency", "status",
]


def _number(value):
    """First integer in a value like 175, '1,200', 'Doors open...\n175 capacity'."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return int(value)
    match = re.search(r"(\d[\d,\s]*)\s*capacity", str(value)) or re.search(r"\d[\d,]*", str(value))
    if not match:
        return None
    digits = re.sub(r"\D", "", match.group(1) if match.lastindex else match.group(0))
    return int(digits) if digits else None


def venue_capacity(event):
    return _number(event.get("capacity")) or None


def to_llm_input(event):
    data = {}
    for field in PLAIN_FIELDS:
        value = event.get(field)
        if value in (None, "", [], {}):
            continue
        if field == "description":
            value = re.sub(r"\s+", " ", str(value)).strip()[:DESCRIPTION_MAX_CHARS]
        data[field] = value

    if data.get("venue_address") == data.get("venue_name"):
        data.pop("venue_address", None)  # same text twice wastes tokens
    # StubHub currently puts the price ("From $114") in venue_name - don't
    # show the model a price as a venue (the price is still sent as price_from).
    if re.match(r"^\s*from\s*[$€£]", str(data.get("venue_name", "")), re.I):
        data.pop("venue_name", None)
    if event.get("date"):
        try:
            data["weekday"] = date.fromisoformat(event["date"]).strftime("%A")
        except ValueError:
            pass

    genres = event.get("genres")
    if isinstance(genres, str):
        try:
            genres = json.loads(genres)
        except ValueError:
            genres = [genres]
    if genres:
        data["genres"] = genres

    capacity = venue_capacity(event)
    if capacity:
        data["venue_capacity"] = capacity
    rsvps = _number(event.get("attendees"))
    if rsvps:
        data["rsvp_count"] = rsvps
    followers = _number(event.get("follow_count"))
    if followers:
        data["performer_followers"] = followers
    return data


# Example: what is sent for the first sample event
print(json.dumps(to_llm_input(sample[0]), indent=2, ensure_ascii=False))

## 3. The prompt

In [ ]:
SYSTEM_PROMPT = """You estimate how many people will physically attend events. The numbers feed a city model of crowds, footfall and demand around venues.

For each event, estimate expectedAttendance: the number of people actually present at the event on that date (audience/spectators plus participants; not staff). For a multi-day event, give the attendance on a typical single day.

How to reason:
1. Work out what the event is: professional or amateur sport (and its level), concert, club night, festival, conference, workshop, talk, meetup, exhibition, guided tour, religious event, etc.
2. Estimate the venue's realistic capacity from its name/address and your knowledge of the city. If "venue_capacity" is given, it is the venue's real maximum: attendance must not exceed it.
3. Estimate how full it will be from demand signals: how well known the artist/team/organiser is, the competition level, price, performer_followers, weekday and time.
4. If "rsvp_count" is given (Meetup RSVPs), use it as the anchor: real turnout is usually 50-90% of RSVPs.

Approximate Paris reference points:
- Stade de France ~80,000; Parc des Princes ~48,000; Paris La Defense Arena ~40,000 (concerts); Accor Arena (Bercy) ~20,000; Zenith Paris ~6,300; Olympia ~2,000; Bataclan ~1,500; La Cigale ~1,400; small clubs and bars 50-300; jazz clubs 50-150.
- Professional handball/basketball/volleyball (Starligue, Betclic Elite, Ligue A): 1,500-5,000. National 1-3 amateur senior leagues: 50-400. Regional/departmental leagues and cups: 20-150. Youth (U13-U18) and low divisions/leisure leagues (e.g. "Division 7", "4x4"): 10-60.
- Eventbrite-style workshops, talks and networking: 10-80. Meetup groups: 10-50. Guided walks/tours: 10-30.

Rules:
- Online events (venue "Online event", webinar, livestream) have nobody physically present: expectedAttendance = 0, low = 0, high = 0.
- Do not inflate. If the event, performer or organiser is not recognisable, assume a small event suited to the venue type.
- "low" and "high" are a plausible range (about 80% sure the real number is inside); low <= expectedAttendance <= high.
- "confidence": "high" = known venue capacity and a clear demand signal; "medium" = venue or event type is clear but demand is uncertain; "low" = mostly guessing.
- "venueCapacityGuess": your estimate of the venue's maximum capacity, or null if you cannot tell.
- "reason": one short sentence (max 25 words) naming the key evidence.

Return ONLY a JSON object, exactly in this shape, with one entry per input event and the same ids:
{"estimates": [{"id": 1, "expectedAttendance": 120, "low": 60, "high": 200, "confidence": "medium", "venueCapacityGuess": 300, "reason": "..."}]}"""


def build_user_prompt(batch):
    """batch: list of (id, llm_input) pairs."""
    items = [{"id": i, **data} for i, data in batch]
    return (
        f"Estimate attendance for these {len(items)} events. Return JSON only, one estimate per id.\n\n"
        + json.dumps(items, ensure_ascii=False, indent=1)
    )


print(SYSTEM_PROMPT)

## 4. Calling Groq — batched, cached, validated

- One request carries `BATCH_SIZE` events; the model must return one estimate per `id`.
- `temperature=0` + JSON mode for stable, parseable answers; rate limits and invalid-JSON replies are retried.
- Online events (venue like "Online event") get 0 by rule and are never sent to the model.
- Each answer is checked: whole numbers, `low <= expected <= high`, never above a known `venue_capacity`. Events the model skipped are not cached, so re-running retries just those.

In [ ]:
CONFIDENCE_LEVELS = {"low", "medium", "high"}
CACHE = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}
USAGE = {}  # model -> requests / tokens / seconds used in this session


def save_cache():
    CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False, indent=1), encoding="utf-8")


def cache_key(model, llm_input, version=None):
    parts = [version or PROMPT_VERSION, model, llm_input]
    if model in MODEL_OPTIONS:  # different settings -> different answers -> separate cache entries
        parts.append(MODEL_OPTIONS[model])
    blob = json.dumps(parts, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


class DailyTokenLimit(Exception):
    """Free-tier daily token cap reached for a model - waiting minutes won't help."""


def _request(model, messages, max_attempts=8):
    kwargs = dict(model=model, messages=messages, temperature=0, response_format={"type": "json_object"})
    if model in MODEL_OPTIONS:
        kwargs.update(MODEL_OPTIONS[model])
    elif model.startswith("openai/gpt-oss"):
        kwargs["reasoning_effort"] = "medium"

    for attempt in range(max_attempts):
        try:
            return client.chat.completions.create(**kwargs)
        except groq.RateLimitError as error:
            if "tokens per day" in str(error):
                raise DailyTokenLimit(f"{model}: daily token limit reached. Groq said: {error}") from None
            if "Request too large" in str(error):
                # Retrying can't help: one request alone exceeds a per-minute limit.
                raise RuntimeError(
                    f"{model}: one batch is bigger than this model's per-minute token limit - "
                    f"lower BATCH_SIZES['{model}']. Groq said: {error}"
                ) from None
            wait = min(90.0, float(error.response.headers.get("retry-after") or 5 * 2**attempt))
            print(f"  rate limited - waiting {wait:.0f}s")
            time.sleep(wait)
        except groq.BadRequestError as error:
            message = str(error)
            if "Request too large" in message:
                raise
            if "json_validate_failed" in message and attempt < max_attempts - 1:
                print("  model returned invalid JSON - retrying")
                continue
            if "reasoning" in message and ("reasoning_effort" in kwargs or "reasoning_format" in kwargs):
                kwargs.pop("reasoning_effort", None)
                kwargs.pop("reasoning_format", None)
                print("  model rejected the reasoning option - retrying without it")
                continue
            raise
        except (groq.APIConnectionError, groq.InternalServerError) as error:
            print(f"  {type(error).__name__} - retrying")
            time.sleep(2**attempt)
    raise RuntimeError(f"{model}: no usable response after {max_attempts} attempts")


def _to_int(value):
    try:
        return max(0, int(round(float(value))))
    except (TypeError, ValueError):
        return None


def clean_estimate(raw, event):
    expected = _to_int(raw.get("expectedAttendance"))
    if expected is None:
        return None
    low, high = _to_int(raw.get("low")), _to_int(raw.get("high"))
    low = expected if low is None else min(low, expected)
    high = expected if high is None else max(high, expected)
    capacity = venue_capacity(event)
    if capacity:  # a real capacity is a hard ceiling
        expected, low, high = min(expected, capacity), min(low, capacity), min(high, capacity)
    confidence = str(raw.get("confidence", "")).lower()
    return {
        "expectedAttendance": expected,
        "low": low,
        "high": high,
        "confidence": confidence if confidence in CONFIDENCE_LEVELS else "low",
        "venueCapacityGuess": _to_int(raw.get("venueCapacityGuess")),
        "reason": str(raw.get("reason") or "").strip(),
    }


ONLINE_RE = re.compile(r"\b(online event|online|webinar|livestream|live stream|zoom)\b", re.I)
ONLINE_ESTIMATE = {
    "expectedAttendance": 0, "low": 0, "high": 0, "confidence": "high",
    "venueCapacityGuess": None, "reason": "Online event - nobody physically present (rule, no LLM call).",
}


def is_online(event):
    """Online-only events never reach the model (gpt-oss-120b sometimes anchored on
    their RSVP count instead of answering 0)."""
    return bool(ONLINE_RE.search(str(event.get("venue_name") or "")))


def v1_rule(event):
    """Answer without the model when a rule settles it (v1: online events only)."""
    return dict(ONLINE_ESTIMATE) if is_online(event) else None


def estimate_attendance(
    events, model=MODEL, batch_size=None, verbose=True,
    version=None, system_prompt=None, input_fn=None, user_prompt_fn=None, clean_fn=None, rule_fn=None,
):
    """Estimate dicts (or None where the model gave nothing usable), aligned with `events`.
    The keyword options let a later prompt version reuse this loop; their defaults are v1."""
    batch_size = batch_size or BATCH_SIZES.get(model, BATCH_SIZE)
    version, system_prompt = version or PROMPT_VERSION, system_prompt or SYSTEM_PROMPT
    input_fn, user_prompt_fn = input_fn or to_llm_input, user_prompt_fn or build_user_prompt
    clean_fn, rule_fn = clean_fn or clean_estimate, rule_fn or v1_rule

    ruled = [rule_fn(e) for e in events]
    inputs = [input_fn(e) for e in events]
    keys = [cache_key(model, d, version) for d in inputs]
    todo = [i for i, k in enumerate(keys) if k not in CACHE and ruled[i] is None]
    usage = USAGE.setdefault(model, {"requests": 0, "prompt_tokens": 0, "completion_tokens": 0, "seconds": 0.0})
    if verbose:
        print(f"{model}: {len(events) - len(todo)} cached, {len(todo)} to estimate")

    for start in range(0, len(todo), batch_size):
        chunk = todo[start : start + batch_size]
        batch = [(n, inputs[i]) for n, i in enumerate(chunk, start=1)]  # ids 1..k inside each request
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_fn(batch)},
        ]
        started = time.time()
        try:
            response = _request(model, messages)
        except DailyTokenLimit as error:
            print(f"  STOPPED: {error}")
            print(f"  {len(todo) - start} events left without an estimate (cached answers are kept) - re-run later")
            break
        usage["requests"] += 1
        usage["seconds"] += time.time() - started
        if response.usage:
            usage["prompt_tokens"] += response.usage.prompt_tokens
            usage["completion_tokens"] += response.usage.completion_tokens

        try:
            payload = json.loads(response.choices[0].message.content or "{}")
        except json.JSONDecodeError:
            payload = {}
        items = payload.get("estimates", []) if isinstance(payload, dict) else []
        by_id = {_to_int(item.get("id")): item for item in items if isinstance(item, dict)}

        ok = 0
        for n, i in enumerate(chunk, start=1):
            cleaned = clean_fn(by_id[n], events[i]) if n in by_id else None
            if cleaned:
                CACHE[keys[i]] = cleaned
                ok += 1
        save_cache()
        if verbose:
            print(f"  batch {start // batch_size + 1}: {ok}/{len(chunk)} estimated in {time.time() - started:.1f}s")

    return [r if r is not None else CACHE.get(k) for r, k in zip(ruled, keys)]


def hints(event):
    """The extra signals a source gave, shown next to the estimate."""
    data = to_llm_input(event)
    keys = ("tournament", "venue_capacity", "rsvp_count", "performer_followers", "price_from", "price_min", "genres")
    return ", ".join(f"{k}={data[k]}" for k in keys if k in data)[:90]


def results_table(events, estimates):
    rows = []
    for e, est in zip(events, estimates):
        est = est or {}
        rows.append(
            {
                "source": e.get("source"),
                "event": (e.get("event_name") or "")[:60],
                "date": e.get("date"),
                "venue": (e.get("venue_name") or "")[:40],
                "hints": hints(e),
                "expected": est.get("expectedAttendance"),
                "low": est.get("low"),
                "high": est.get("high"),
                "confidence": est.get("confidence"),
                "venue_cap_guess": est.get("venueCapacityGuess"),
                "reason": est.get("reason"),
            }
        )
    return pd.DataFrame(rows)

## 5. Run on the test sample

In [ ]:
estimates = estimate_attendance(sample, MODEL)
df = results_table(sample, estimates)
print("usage this session:", USAGE.get(MODEL))
df

In [ ]:
# Spread of estimates per source
df.groupby("source")["expected"].describe()[["count", "min", "50%", "max"]]

## 6. Sanity check — Meetup events with real RSVP counts

Turnout should land around 50-90% of the RSVPs.

In [ ]:
df[df["source"] == "meetup"][["event", "hints", "expected", "low", "high", "reason"]]

## 7. Compare models on the same events

Qwen runs 3 events per request at about 1 request/minute (free-tier limit), so its first run takes ~15 minutes; after that it comes from the cache.

`max/min` = how far apart the models are. Rows near 1 = they agree (more trustworthy); big ratios = worth a closer look.

In [ ]:
short = {m: m.split("/")[-1] for m in COMPARE_MODELS}
comparison = pd.DataFrame(
    {"source": [e.get("source") for e in sample], "event": [(e.get("event_name") or "")[:55] for e in sample]}
)
for m in COMPARE_MODELS:
    comparison[short[m]] = [(x or {}).get("expectedAttendance") for x in estimate_attendance(sample, m)]

cols = list(short.values())
comparison["max/min"] = (comparison[cols].max(axis=1) / comparison[cols].min(axis=1).clip(lower=1)).round(1)
comparison.sort_values("max/min", ascending=False)

In [ ]:
# Requests, tokens and time per model in this session (cached answers cost nothing)
pd.DataFrame(USAGE).T

## 8. Try any event by name

In [ ]:
def estimate_by_name(text, model=MODEL, limit=5):
    """Estimate the first `limit` Paris events whose name contains `text`."""
    matches = [e for e in all_events if text.lower() in (e.get("event_name") or "").lower()][:limit]
    if not matches:
        print("no Paris event matches", repr(text))
        return None
    return results_table(matches, estimate_attendance(matches, model))


estimate_by_name("jazz")

## 9. Validation against real-world numbers

`notebooks/attendance_ground_truth.json` holds what could be verified online for the test sample (researched 2026-11-20, each row with its source URL):
actual attendance where any was published (rare), a typical crowd (Rennes volleyball, from official LNV match reports), and the **real venue capacity** (national sports-facility register, venue sites, Wikipedia).

Two checks per model: does the estimate stay **below the real capacity**, and how close is the model's **`venueCapacityGuess`** to the real capacity.

In [ ]:
TRUTH_PATH = PROJECT_ROOT / "notebooks" / "attendance_ground_truth.json"
truth = {t["canonical_key"]: t for t in json.loads(TRUTH_PATH.read_text(encoding="utf-8"))}
by_key = {e["_key"]: e for e in all_events}
truth_events = [by_key[k] for k in truth if k in by_key]
short = {m: m.split("/")[-1] for m in COMPARE_MODELS}


def validation_table(models=COMPARE_MODELS):
    estimates = {m: estimate_attendance(truth_events, m, verbose=False) for m in models}
    rows = []
    for n, e in enumerate(truth_events):
        t = truth[e["_key"]]
        row = {
            "source": e.get("source"),
            "event": (e.get("event_name") or "")[:40],
            "actual": t["actual_attendance"],
            "typical": t["typical_attendance"],
            "real_capacity": t["venue_capacity"],
        }
        for m in models:
            est = estimates[m][n] or {}
            row[short[m]] = est.get("expectedAttendance")
            row[short[m] + "_cap"] = est.get("venueCapacityGuess")
        row["notes"] = (t.get("notes") or "")[:70]
        rows.append(row)
    return pd.DataFrame(rows)


val = validation_table()
val

In [ ]:
def score(val, models=COMPARE_MODELS):
    known = val[val["real_capacity"].fillna(0) > 0]
    known = known[known["actual"].ne(0)]  # skip cancelled / online rows
    result = {}
    for m in models:
        s = short[m]
        ratio = known[s + "_cap"].astype(float) / known["real_capacity"]
        result[s] = {
            "events with real capacity": len(known),
            "estimate ABOVE real capacity": int((known[s] > known["real_capacity"]).sum()),
            "capacity guess within 0.67x-1.5x": int(ratio.between(0.67, 1.5).sum()),
            "median capacity-guess error %": round(float((ratio - 1).abs().median()) * 100),
        }
    return pd.DataFrame(result)


score(val)

In [ ]:
# The few events with a real attendance figure
val[val["actual"].notna() & val["actual"].ne(0) | val["typical"].notna() | (val["notes"].str.contains("CANCELLED"))]

# Version 2 - real capacities, location/status rules, stricter prompt

What changes compared with v1 (sections 1-9 above stay as they were, for comparison):

1. **Real venue capacity** for sports venues from France's national sports-facility register (`equipements.sports.gouv.fr`, free API, field `equip_trib_nb` = fixed stand seats). DICE's capacity is still used too. A verified capacity is a hard ceiling.
2. **Rules before the model** (no LLM call, no tokens):
   - online event -> 0
   - status says cancelled/postponed -> 0
   - outside the city -> skipped: postcode outside the city, or a StubHub URL whose slug names another city (most "Paris" StubHub events are really New York area)
3. **Stricter prompt**: participants are counted for amateur sport, one performance for long runs, estimate never above the model's own capacity guess, `likelyInCity` flag (catches e.g. a Rennes home match placed in Paris), `eventKind`, and a stricter `confidence`.
4. **Model: `openai/gpt-oss-120b`** (best in the section 9 validation).

In [ ]:
import unicodedata

import requests

TARGET_CITY = "Paris"
MODEL_V2 = "openai/gpt-oss-120b"
PROMPT_VERSION_V2 = "v2"

# Per-city settings for the location rules and the register search.
CITY_RULES = {
    "Paris": {"postcode_prefixes": ("75",), "department": "75", "url_slug": "paris"},
    "Lyon": {"postcode_prefixes": ("69",), "department": "69", "url_slug": "lyon"},
}

REGISTER_API = "https://equipements.sports.gouv.fr/api/explore/v2.1/catalog/datasets/data-es/records"
REGISTER_CACHE_PATH = PROJECT_ROOT / "notebooks" / "venue_capacity_cache.json"
REGISTER_CACHE = (
    json.loads(REGISTER_CACHE_PATH.read_text(encoding="utf-8")) if REGISTER_CACHE_PATH.exists() else {}
)

## 10. Real capacity from the sports-facility register

Venue names from the sports API are messy (`"Chateau des Rentiers - 20h - Paris13"`, `"Paris Bercy - Sonja Henie"`), so the lookup:
strips generic words (gymnase, centre sportif, Paris, times...), searches facility + equipment names inside the venue's postcode (or the city's department), and only accepts equipment that suits the sport (an ice rink for hockey, an indoor hall for handball/basketball/volleyball...).
A single short word without a postcode is never searched (too vague). `0` or missing stand seats = unknown (many gyms have no fixed stands recorded).

In [ ]:
_GENERIC_WORDS = {
    "GYMNASE", "CENTRE", "SPORTIF", "SPORTIVE", "COMPLEXE", "SALLE", "SALLES", "STADE", "PARIS", "DE", "DU",
    "DES", "LA", "LE", "LES", "EX", "ET", "D", "L", "SUR", "PATINOIRE", "PISCINE", "ESPACE", "TERRAIN", "CITY",
}
# Words in a facility's type/name that make it suitable for a sport (default: an indoor hall).
_SPORT_FACILITY_WORDS = {
    "ice-hockey": ("patinoire", "glace"),
    "football": ("grands jeux", "football"),
    "rugby": ("grands jeux", "rugby"),
    "american-football": ("grands jeux", "football americain"),
    "beach-soccer": ("sable", "plage"),
    "baseball": ("baseball", "softball"),
    "softball": ("baseball", "softball"),
}
_INDOOR_WORDS = ("salle", "gymnase", "dojo", "palais", "arena")


def _ascii_upper(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().upper()


def _venue_words(venue):
    text = _ascii_upper(venue)
    text = re.sub(r"[()]", " ", text)
    text = re.sub(r"\b\d{1,2}H\d{0,2}\b|\bPARIS\s?\d+E?\b|\b\d+E\b", " ", text)  # times, "Paris13", "16e"
    return [w for w in re.split(r"[^A-Z0-9]+", text) if len(w) > 1 and not w.isdigit() and w not in _GENERIC_WORDS]


def register_capacity(event, city=TARGET_CITY):
    # Fixed stand seats from the national register for a sports event's venue, or None.
    if event.get("source") != "sportsevents24" or not event.get("venue_name"):
        return None
    sport = (event.get("type") or "").lower()
    address = str(event.get("venue_address") or "")
    cache_id = json.dumps([event["venue_name"], address, sport, city], ensure_ascii=False)
    if cache_id in REGISTER_CACHE:
        return REGISTER_CACHE[cache_id]

    words = _venue_words(event["venue_name"])
    postcode = re.search(r"\b(\d{5})\b", address)
    where_place = f'inst_cp="{postcode.group(1)}"' if postcode else f'dep_code="{CITY_RULES[city]["department"]}"'
    suitable = _SPORT_FACILITY_WORDS.get(sport, _INDOOR_WORDS)
    queries = [" ".join(words)] + sorted(words, key=len, reverse=True)[:2] if words else []

    found = None
    for query in dict.fromkeys(queries):
        if " " not in query and not postcode and len(query) < 5:
            continue  # a short single word without a postcode is too vague
        name_match = " and ".join(
            f"(search(inst_nom, {json.dumps(w)}) or search(equip_nom, {json.dumps(w)}))" for w in query.split()
        )
        params = {
            "where": f"({name_match}) and {where_place}",
            "select": "equip_numero,inst_nom,equip_nom,equip_type_name,equip_type_famille,equip_trib_nb",
            "limit": 50,
        }
        try:
            rows = requests.get(REGISTER_API, params=params, timeout=20).json().get("results", [])
        except (requests.RequestException, ValueError):
            rows = []
        fitting = [
            r for r in rows
            if any(
                w in _ascii_upper(f'{r["equip_type_name"]} {r["equip_type_famille"]} {r["equip_nom"]}').lower()
                for w in suitable
            )
        ]
        if fitting:
            # Most stand seats wins; on a tie prefer a real gymnasium/hall.
            best = max(
                fitting, key=lambda r: (r.get("equip_trib_nb") or 0, "gymnase" in r["equip_type_name"].lower())
            )
            found = {
                "seats": best.get("equip_trib_nb") or None,  # 0 = not recorded / no fixed stands
                "facility": f'{best["inst_nom"]} - {best["equip_nom"]}',
                "equip_numero": best["equip_numero"],
                "matched_query": query,
            }
            break

    REGISTER_CACHE[cache_id] = found
    REGISTER_CACHE_PATH.write_text(json.dumps(REGISTER_CACHE, ensure_ascii=False, indent=1), encoding="utf-8")
    return found


def verified_capacity(event):
    # (capacity, where it comes from) from real data, or (None, None).
    dice = venue_capacity(event)
    if dice:
        return dice, "ticketing page (DICE)"
    reg = register_capacity(event)
    if reg and reg["seats"]:
        return reg["seats"], f'French sports-facility register: fixed stand seats ({reg["facility"]})'
    return None, None


sports_sample = [e for e in sample if e.get("source") == "sportsevents24"]
pd.DataFrame(
    [
        {"venue": e["venue_name"][:38], "sport": e.get("type"), **(register_capacity(e) or {"facility": "- no match -"})}
        for e in sports_sample
    ]
)

## 11. Rules before the model: online, cancelled, outside the city

In [ ]:
CANCELLED_RE = re.compile(r"cancel|annul|postpon|report", re.I)
_STUBHUB_SLUG_RE = re.compile(r"stubhub\.[a-z.]+/([^/?#]+)-tickets-\d{1,2}-\d{1,2}-\d{4}", re.I)


def location_problem(event, city=TARGET_CITY):
    # Why the event is clearly NOT in `city`, or None.
    rules = CITY_RULES[city]
    slug = _STUBHUB_SLUG_RE.search(str(event.get("url") or ""))
    if slug and rules["url_slug"] not in slug.group(1).lower():
        return f'StubHub URL names another place ("{slug.group(1)}")'
    postcode = re.search(r"\b(\d{5})\b", str(event.get("venue_address") or ""))
    if postcode and not postcode.group(1).startswith(rules["postcode_prefixes"]):
        return f"postcode {postcode.group(1)} is outside {city}"
    return None


def _ruled(status, expected, reason):
    return {
        "status": status, "expectedAttendance": expected, "low": expected, "high": expected,
        "confidence": "high" if expected == 0 else None, "venueCapacityGuess": None,
        "likelyInCity": None, "eventKind": None, "capacityUsed": None, "capacitySource": None,
        "reason": reason + " (rule, no LLM call)",
    }


def rule_v2(event, city=TARGET_CITY):
    if is_online(event):
        return _ruled("online", 0, "Online event - nobody physically present")
    if CANCELLED_RE.search(str(event.get("status") or "")):
        return _ruled("cancelled", 0, f'Status is "{event.get("status")}"')
    problem = location_problem(event, city)
    if problem:
        return _ruled("outside_city", None, f"Skipped: {problem}")
    return None


# How many Paris events each rule would take out before any LLM call
counts = pd.Series([(rule_v2(e) or {"status": "to LLM"})["status"] for e in all_events]).value_counts()
counts

In [ ]:
# Where the "Paris" StubHub events really are (city part of the URL slug, top 10)
stubhub_places = pd.Series(
    [
        "-".join(m.group(1).split("-")[-2:])
        for e in all_events
        if e.get("source") == "stubhub" and (m := _STUBHUB_SLUG_RE.search(str(e.get("url") or "")))
    ]
).value_counts()
stubhub_places.head(10)

## 12. Prompt v2

In [ ]:
SYSTEM_PROMPT_V2 = '''You estimate how many people will physically attend events. The numbers feed a city model of crowds, footfall and demand around venues. All events below are listed for {CITY}.

For each event, estimate expectedAttendance: the people physically present at ONE occurrence of the event - audience/spectators plus participants (players, coaches, referees, performers, speakers); not venue staff. For an exhibition or a show that runs on many dates, estimate one day / one performance.

How to reason:
1. Work out what the event is (see eventKind below) and, for sport, its level.
2. Venue capacity. If "venue_capacity" is given it is VERIFIED real data (its origin is in "venue_capacity_source"): attendance must not exceed it. For sports halls it is the number of fixed stand seats - use it as the ceiling. If it is not given, estimate the capacity yourself (venueCapacityGuess). Big venue names can refer to a small annex (a side hall, a training rink, a small room) - do not assume the main hall's size when the name points to a smaller part. Never return expectedAttendance above your own venueCapacityGuess.
3. Fill rate from demand signals: how well known the artist/team/organiser is, competition level, price, performer_followers, weekday and time. Most small events are far from full.
4. "rsvp_count" (Meetup) is the strongest signal: real turnout is usually 50-90% of RSVPs and almost never above the RSVP count.

Approximate reference points ({CITY} area):
- Stade de France ~80,000; Parc des Princes ~48,000; Paris La Defense Arena ~40,000 (concerts); Accor Arena main hall ~20,000; Zenith Paris ~6,300; Olympia ~2,000; Bataclan ~1,500; La Cigale ~1,400; theatres 300-1,000; small clubs and bars 50-300; jazz clubs 50-150; chapels/small churches 80-300.
- Amateur sport: count both teams (players, coaches, referees: about 25-35 people) plus spectators. Youth (U11-U18), corporate/leisure leagues ("Division 6", "4x4"), departmental and regional leagues and cups: 30-80 in total. National 2-3 senior amateur: 80-300. Professional second tier of indoor sports (e.g. Ligue B, Pro B, Proligue): 400-1,500. Top tier indoor (Starligue, Betclic Elite, Ligue A): 1,500-5,000.
- Workshops, talks, networking, afterworks: 10-80. Meetup groups: 10-50. Guided walks/visits: 10-30 per group. Film screenings in a cinema: 20-40% of seats.

Rules:
- Do not inflate. If the event, performer or organiser is not recognisable, assume a small event suited to the venue type.
- "low" and "high": a plausible range (about 80% sure the real number is inside); low <= expectedAttendance <= high.
- "likelyInCity": "no" if the event clearly takes place somewhere else (e.g. the home team, venue or address belongs to another city or country), "unsure" if you cannot tell, otherwise "yes".
- "confidence": "high" ONLY when a verified venue_capacity or an rsvp_count is given AND the event type is clear; "medium" when you know the venue or the event type gives a narrow range; otherwise "low". Most small or unfamiliar events should be "low".
- "eventKind": one of pro_sport, amateur_sport, concert, club_night, theatre_show, festival, exhibition, conference_talk, workshop_class, meetup_networking, tour_visit, film_screening, religious, other.
- "venueCapacityGuess": your estimate of the venue's maximum capacity (repeat venue_capacity when given), or null if you cannot tell.
- "reason": one short sentence (max 25 words) naming the key evidence.

Return ONLY a JSON object in exactly this shape, one entry per input event, same ids:
{"estimates": [{"id": 1, "expectedAttendance": 60, "low": 40, "high": 90, "confidence": "low", "venueCapacityGuess": 200, "likelyInCity": "yes", "eventKind": "amateur_sport", "reason": "..."}]}'''.replace("{CITY}", TARGET_CITY)


def to_llm_input_v2(event):
    data = to_llm_input(event)
    capacity, origin = verified_capacity(event)
    if capacity:
        data["venue_capacity"] = capacity
        data["venue_capacity_source"] = origin
    return data


def build_user_prompt_v2(batch):
    items = [{"id": i, **data} for i, data in batch]
    return (
        f"Estimate attendance for these {len(items)} events listed for {TARGET_CITY}. "
        "Return JSON only, one estimate per id.\n\n" + json.dumps(items, ensure_ascii=False, indent=1)
    )


print(SYSTEM_PROMPT_V2)

## 13. Checking each v2 answer

On top of v1's checks: a **verified capacity** (DICE or register) is a hard ceiling, otherwise the **model's own capacity guess** is the ceiling. `capacityUsed` / `capacitySource` record which one applied.

In [ ]:
LIKELY_IN_CITY = {"yes", "no", "unsure"}


def clean_estimate_v2(raw, event):
    est = clean_estimate(raw, event)  # v1 checks (whole numbers, low <= expected <= high, DICE capacity)
    if est is None:
        return None
    capacity, origin = verified_capacity(event)
    if capacity is None and est["venueCapacityGuess"]:
        capacity, origin = est["venueCapacityGuess"], "model's own capacity guess"
    if capacity:
        for field in ("expectedAttendance", "low", "high"):
            est[field] = min(est[field], capacity)
    in_city = str(raw.get("likelyInCity", "")).lower()
    est.update(
        status="estimated",
        likelyInCity=in_city if in_city in LIKELY_IN_CITY else "unsure",
        eventKind=str(raw.get("eventKind") or "other"),
        capacityUsed=capacity,
        capacitySource=origin,
    )
    return est


def estimate_attendance_v2(events, model=MODEL_V2, verbose=True):
    return estimate_attendance(
        events, model, verbose=verbose, version=PROMPT_VERSION_V2, system_prompt=SYSTEM_PROMPT_V2,
        input_fn=to_llm_input_v2, user_prompt_fn=build_user_prompt_v2, clean_fn=clean_estimate_v2, rule_fn=rule_v2,
    )


def results_table_v2(events, estimates):
    rows = []
    for e, est in zip(events, estimates):
        est = est or {}
        rows.append(
            {
                "source": e.get("source"),
                "event": (e.get("event_name") or "")[:45],
                "venue": (e.get("venue_name") or "")[:30],
                "status": est.get("status"),
                "expected": est.get("expectedAttendance"),
                "low": est.get("low"),
                "high": est.get("high"),
                "confidence": est.get("confidence"),
                "capacity": est.get("capacityUsed"),
                "capacity_from": (est.get("capacitySource") or "")[:32],
                "in_city": est.get("likelyInCity"),
                "kind": est.get("eventKind"),
                "reason": est.get("reason"),
            }
        )
    return pd.DataFrame(rows)

## 14. Run v2 on the test sample

In [ ]:
estimates_v2 = estimate_attendance_v2(sample)
df_v2 = results_table_v2(sample, estimates_v2)
print("usage this session:", USAGE.get(MODEL_V2))
df_v2

In [ ]:
# Events the model thinks are NOT in the city (kept, but worth excluding or checking)
df_v2[df_v2["in_city"].isin(["no", "unsure"])][["source", "event", "venue", "in_city", "expected", "reason"]]

In [ ]:
# Is "confidence" more informative now?
pd.DataFrame({"v1 (120b)": df_conf_v1, "v2 (120b)": df_v2["confidence"].value_counts()}) if (
    df_conf_v1 := pd.Series([(x or {}).get("confidence") for x in estimate_attendance(sample, MODEL_V2, verbose=False)]).value_counts()
) is not None else None

## 15. Validation: v1 vs v2 against the real-world numbers

Same ground truth as section 9. Note: for sports venues v2 *uses* the register, which is also where most of the real sports capacities came from - so for those rows the capacity check is no longer independent; that is the point of v2 (use real data instead of guessing).

In [ ]:
def compare_versions(model=MODEL_V2):
    v1 = estimate_attendance(truth_events, model, verbose=False)
    v2 = estimate_attendance_v2(truth_events, model, verbose=False)
    rows = []
    for e, a, b in zip(truth_events, v1, v2):
        t = truth[e["_key"]]
        a, b = a or {}, b or {}
        rows.append(
            {
                "source": e.get("source"),
                "event": (e.get("event_name") or "")[:38],
                "actual": t["actual_attendance"],
                "typical": t["typical_attendance"],
                "real_capacity": t["venue_capacity"],
                "v1": a.get("expectedAttendance"),
                "v1_cap": a.get("venueCapacityGuess"),
                "v2": b.get("expectedAttendance"),
                "v2_cap": b.get("capacityUsed"),
                "v2_status": b.get("status"),
                "v2_in_city": b.get("likelyInCity"),
                "notes": (t.get("notes") or "")[:55],
            }
        )
    return pd.DataFrame(rows)


cmp_val = compare_versions()


def score_versions(table):
    known = table[(table["real_capacity"].fillna(0) > 0) & table["actual"].ne(0)]
    result = {}
    for v in ("v1", "v2"):
        rows = known[known[v].notna()]
        ratio = rows[v + "_cap"].astype(float) / rows["real_capacity"]
        result[v] = {
            "events scored (real capacity known)": len(rows),
            "estimate ABOVE real capacity": int((rows[v] > rows["real_capacity"]).sum()),
            "capacity within 0.67x-1.5x": int(ratio.between(0.67, 1.5).sum()),
            "median capacity error %": round(float((ratio - 1).abs().median()) * 100),
            "skipped by rules (not scored)": int(known[v].isna().sum()),
        }
    return pd.DataFrame(result)


score_versions(cmp_val)

In [ ]:
# Events with a real attendance figure, plus everything v2 skipped or flagged
cmp_val[
    cmp_val["actual"].notna() & cmp_val["actual"].ne(0)
    | cmp_val["typical"].notna()
    | cmp_val["notes"].str.contains("CANCELLED")
    | cmp_val["v2_status"].ne("estimated") & cmp_val["v2_status"].ne("online")
    | cmp_val["v2_in_city"].isin(["no", "unsure"])
]

# Version 3 - tuned to your labels

`notebooks/attendance_labels_filled.csv` holds your 53 labelled events (attendance + radius). This part:

1. **Scores v2** against the labels (baseline).
2. **Radius in code**: the rubric's radius tiers, as a function of attendance (no LLM tokens, same answer every time).
3. **Prompt v3**: the labelling prompt's definitions, fill-rate rubric, category priors and confidence rules, plus v2's verified capacities and rules. Rounding is done in code.
4. **v3 + priors**: the model names a category prior, code clamps attendance into its range.
5. **+ web search** (`browser_search`) only for the biggest concerts / pro sport (free tier: ~3 searches a day).
6. Side-by-side scores and cost.

**Fair scoring**: rows 9, 12, 13, 32 and 47 are the labelling prompt's own examples (their answers are written in it), so they are **not scored**. Some category priors were written with these rows in mind, so the scores are a bit optimistic - check the final version on a fresh batch of labelled events.

## 16. Load your labels

In [ ]:
import math

LABELS_PATH = PROJECT_ROOT / "notebooks" / "attendance_labels_filled.csv"
EXAMPLE_ROWS = {9, 12, 13, 32, 47}  # answers appear in the labelling prompt -> not scored

labels = pd.read_csv(LABELS_PATH, encoding="utf-8-sig")
by_key = {e["_key"]: e for e in all_events}
missing = labels.loc[~labels["canonical_key"].isin(by_key.keys()), "label_id"].tolist()
labels = labels[labels["canonical_key"].isin(by_key.keys())].reset_index(drop=True)
label_events = [by_key[k] for k in labels["canonical_key"]]

# One comparison table, filled in step by step below.
lab = pd.DataFrame(
    {
        "id": labels["label_id"],
        "event": labels["event_name"].str[:34],
        "scored": ~labels["label_id"].isin(EXAMPLE_ROWS),
        "label": labels["YOUR_expected_attendance"],
        "label_radius": labels["YOUR_radius_meters"],
    }
)
print(f"{len(labels)} labelled events found in the DB (missing label_ids: {missing or 'none'}); "
      f"{lab['scored'].sum()} scored, {len(EXAMPLE_ROWS)} held out")
lab.head()

In [ ]:
def agreement(pred, truth):
    """How close predictions are to the labels (both pandas Series)."""
    pred, truth = pd.to_numeric(pred, errors="coerce"), pd.to_numeric(truth, errors="coerce")
    ok = pred.notna() & truth.gt(0)
    ratio = pred[ok] / truth[ok]
    n = int(ok.sum())
    if n == 0:
        return {"events scored": 0}
    return {
        "events scored": n,
        "median error %": round(float((ratio - 1).abs().median() * 100)),
        "within ±30%": f"{int(ratio.between(0.7, 1.3).sum())}/{n}",
        "within 2x": f"{int(ratio.between(0.5, 2).sum())}/{n}",
        "bias (median est/label)": round(float(ratio.median()), 2),
    }


def score_columns(table, columns, truth_col="label", mask=None):
    rows = table[table["scored"] & (True if mask is None else mask)]
    return pd.DataFrame({c: agreement(rows[c], rows[truth_col]) for c in columns})


def biggest_misses(table, col, truth_col="label", n=10, extra=()):
    rows = table[table["scored"] & table[col].notna()].copy()
    rows["x label"] = ((rows[col] + 1) / (rows[truth_col] + 1)).round(2)
    rows = rows.assign(_miss=rows["x label"].map(lambda r: abs(math.log(r))))
    return rows.sort_values("_miss", ascending=False).head(n)[["id", "event", truth_col, col, "x label", *extra]]

## 17. Baseline: v2 against your labels

Events from the earlier test sample are already cached; the new ones cost one request per 8 events.

In [ ]:
est_v2_lab = estimate_attendance_v2(label_events)
lab["v2"] = [(e or {}).get("expectedAttendance") for e in est_v2_lab]
lab["v2_status"] = [(e or {}).get("status") for e in est_v2_lab]
score_columns(lab, ["v2"])

In [ ]:
biggest_misses(lab, "v2", extra=["v2_status"])

## 18. Radius from attendance (code, no LLM)

The labelling rubric's radius tiers. Inside a tier the radius grows with log(attendance); rounded to 50 m, minimum 100 m; open-air events get +35% (the rubric says +25-50%).
For comparison: a single formula `k·√attendance` with `k` fitted to your labels.

First check, using **your own attendance labels** as input: this measures only the radius function, not the attendance model.

In [ ]:
# (attendance from, attendance to, radius from, radius to) - the rubric's radius tiers
RADIUS_TIERS = [
    (0, 50, 100, 100), (50, 200, 100, 150), (200, 500, 200, 300), (500, 1500, 300, 500),
    (1500, 3000, 500, 700), (3000, 8000, 800, 1000), (8000, 20000, 1000, 1500), (20000, 80000, 1500, 2500),
]
OUTDOOR_RADIUS_FACTOR = 1.35
MIN_RADIUS = 100


def radius_from_attendance(attendance, outdoor=False):
    if attendance is None or (isinstance(attendance, float) and math.isnan(attendance)):
        return None
    if attendance <= 0:
        return 0
    a = min(attendance, RADIUS_TIERS[-1][1])
    for a0, a1, r0, r1 in RADIUS_TIERS:
        if a < a1 or a1 == RADIUS_TIERS[-1][1]:
            position = 0 if a0 == 0 else math.log(max(a, a0) / a0) / math.log(a1 / a0)
            radius = r0 + (r1 - r0) * position
            break
    if outdoor:
        radius *= OUTDOOR_RADIUS_FACTOR
    return max(MIN_RADIUS, int(round(radius / 50) * 50))


SQRT_K = float((lab["label_radius"] / lab["label"] ** 0.5)[lab["scored"]].median())


def radius_sqrt(attendance):
    if attendance is None or (isinstance(attendance, float) and math.isnan(attendance)):
        return None
    return 0 if attendance <= 0 else max(MIN_RADIUS, int(round(SQRT_K * attendance**0.5 / 50) * 50))


lab["r_tiers(label)"] = lab["label"].map(radius_from_attendance)
lab["r_sqrt(label)"] = lab["label"].map(radius_sqrt)
print(f"fitted k = {SQRT_K:.1f}  ->  radius = {SQRT_K:.1f} x sqrt(attendance)")
score_columns(lab, ["r_tiers(label)", "r_sqrt(label)"], truth_col="label_radius")

In [ ]:
# Where the radius function alone disagrees with your radius
biggest_misses(lab, "r_tiers(label)", truth_col="label_radius", n=8, extra=["label"])

## 19. Prompt v3

Taken from the labelling prompt: the definition (distinct people on the listed date, sessions added together, one day of a long-running event), "what the event really is", capacity for this configuration, the fill-rate table, the category priors and the confidence rules.
Kept from v2: verified capacities (DICE, sports register) as ceilings, online/cancelled/outside-city rules, `likelyInCity`, `eventKind`, venue reference points.
New output fields: `sessions` (lets attendance exceed one session's capacity, e.g. two guided tours) and `outdoor` (radius +35%).
Rounding (<100 → 5, 100-1,000 → 10, >1,000 → 100) and the radius are done in code.

In [ ]:
PROMPT_VERSION_V3 = "v3"
# Model for v3 and later. gpt-oss-120b is the best; switch to "openai/gpt-oss-20b" when
# 120b's free daily quota (200k tokens) is used up - 20b has its own quota.
MODEL_V3 = MODEL_V2
TODAY = date.today().isoformat()

SYSTEM_PROMPT_V3 = """You are a careful event-analytics estimator. You estimate how many people attend events listed for {CITY}. Your numbers feed a city model of crowds, footfall and demand around venues. Today is {TODAY}.
Consistency matters as much as precision: similar events must get similar numbers unless the data says otherwise. A reasoned estimate with an honest confidence level beats a precise-looking guess.

WHAT TO ESTIMATE
expectedAttendance = distinct people physically present on the listed date: audience + participants (players, coaches, referees, performers, speakers); exclude venue staff.
- Events before today: the actual attendance. Today or later: the expected attendance.
- Multi-hour events with turnover (fairs, open days, open-air parties): total unique people that day.
- Several sessions on the same date (e.g. tours at 10:00 and 14:00): add the sessions together and set "sessions" to their number.
- Multi-day events and long-running exhibitions: the listed date only (that day's visitors).

HOW TO REASON
1. Work out what the event really is. Titles can mislead: a musician's name may be a jam session he joins; a festival name may stand for one act that night; a "museum" may be an ongoing free exhibit.
2. Capacity for THIS configuration (standing vs seated, which room or hall). If "venue_capacity" is given it is verified real data (origin in "venue_capacity_source") and a ceiling - but it may be the main hall of a big complex. Big venue names can refer to a small annex (side hall, training rink, small room). Otherwise estimate the capacity yourself (venueCapacityGuess).
3. Attendance = capacity x fill rate. Use the category prior instead when capacity does not drive turnout (e.g. youth sport in a 4,000-seat arena).

FILL RATE (share of the configuration's capacity)
- The data says sold out / complet: 95-100%
- Extra dates added for demand: 90-98%
- Established act, well-known venue, on sale, no sell-out evidence: 75-90%
- Discounts or promo pushes: 60-80%
- Emerging or unknown act, small club, weeknight: 50-70%
- Monday-Wednesday: subtract 5-10 points. July-August for local-audience events: subtract 10-20 points.

CATEGORY PRIORS (when capacity does not drive turnout)
- Corporate/leisure leagues (e.g. FSGT 4x4 volleyball, "Division 6/7"): players only, 10-16
- Youth sport U13-U17: 25-80 (players and parents), even in a 4,000-seat arena
- Amateur departmental/regional senior matches and cups: 30-80
- 4th-5th tier club sport (e.g. Nationale 2/3): 50-150
- Ice hockey 3rd tier (Division 2): 150-400
- Professional second tier indoor sport (Ligue B, Pro B, Proligue): 400-1,500; top tier indoor (Starligue, Betclic Elite, Ligue A): 1,500-5,000
- Para-sport lower divisions: 20-40
- Free meetups: rsvp_count x 0.75-0.9; events with a registration cap: 85-100% of the cap
- Guided tours: group size (usually 15-30) x number of sessions
- Book talks, museum or library lectures: 40-80; academic half-day seminars: 40-80
- Workshops, networking, afterworks: 15-60
- Free weekly jam sessions / bar nights: 40-60% of capacity
- Free organ recitals in major churches: 200-500
- Free exhibitions: that day's visitors; quiet ones 50-150
- Parish youth groups: 20-50; monastery/religious retreats: 8-20
- Municipal gaming or community fairs: 300-1,000 per day
- Professional network days: typically 150-400

VENUE REFERENCE POINTS ({CITY} area)
Stade de France ~80,000; Parc des Princes ~48,000; Paris La Defense Arena ~40,000 (concerts); Accor Arena ~20,000; Adidas Arena ~8,000; Zenith Paris ~6,300; Olympia ~2,000; Salle Pleyel ~1,900; Bataclan ~1,500; La Cigale ~1,400; Trianon ~1,000; Elysee Montmartre ~1,300; theatres 300-1,000; small clubs and bars 50-300; jazz clubs 50-150; chapels/small churches 80-300.

OUTPUT RULES
- Do not inflate. If the event, performer or organiser is not recognisable, assume a small event suited to the venue type.
- "low" and "high": the range you are about 80% sure contains the real number; low <= expectedAttendance <= high.
- "confidence": "high" only with direct evidence (sold out, registration cap, rsvp_count) AND a verified venue_capacity; "medium" with a verified capacity plus indirect signals, or a well-matched category prior; otherwise "low". When unsure pick the lower level and widen the range.
- "likelyInCity": "no" if the event clearly takes place somewhere else (the home team, venue or address belongs to another city or country), "unsure" if you cannot tell, otherwise "yes".
- "eventKind": one of pro_sport, amateur_sport, concert, club_night, theatre_show, festival, exhibition, conference_talk, workshop_class, meetup_networking, tour_visit, film_screening, religious, fair_expo, other.
- "venueCapacityGuess": the capacity of this configuration (repeat venue_capacity when it applies), or null if you cannot tell.
- "sessions": number of separate sessions on that date that you added together (usually 1).
- "outdoor": true if the event is open-air or mainly outdoors (street, park, walking tour), else false.
- "reason": one short sentence (max 25 words) naming what the event really is and the key evidence.

Return ONLY a JSON object in exactly this shape, one entry per input event, same ids:
{"estimates": [{"id": 1, "expectedAttendance": 60, "low": 40, "high": 90, "confidence": "low", "venueCapacityGuess": 200, "sessions": 1, "outdoor": false, "likelyInCity": "yes", "eventKind": "amateur_sport", "reason": "..."}]}""".replace(
    "{CITY}", TARGET_CITY
).replace("{TODAY}", TODAY)


def build_user_prompt_v3(batch):
    items = [{"id": i, **data} for i, data in batch]
    return (
        f"Today is {TODAY}. Estimate attendance for these {len(items)} events listed for {TARGET_CITY}. "
        "Return JSON only, one estimate per id.\n\n" + json.dumps(items, ensure_ascii=False, indent=1)
    )


print(SYSTEM_PROMPT_V3)

In [ ]:
def round_attendance(n):
    """The rubric's rounding: <100 -> nearest 5, 100-1,000 -> nearest 10, >1,000 -> nearest 100."""
    if not n:
        return n
    step = 5 if n < 100 else 10 if n <= 1000 else 100
    return max(step, int(round(n / step) * step))


def clean_estimate_v3(raw, event, capacity_fn=None):
    est = clean_estimate(raw, event)  # v1 checks (whole numbers, low <= expected <= high, DICE capacity)
    if est is None:
        return None
    sessions = min(max(_to_int(raw.get("sessions")) or 1, 1), 10)
    capacity, origin = (capacity_fn or verified_capacity)(event)
    if capacity is None and est["venueCapacityGuess"]:
        capacity, origin = est["venueCapacityGuess"], "model's own capacity guess"
    if capacity:  # one session can't exceed the capacity; several sessions can
        for field in ("expectedAttendance", "low", "high"):
            est[field] = min(est[field], capacity * sessions)
    for field in ("expectedAttendance", "low", "high"):
        est[field] = round_attendance(est[field])
    outdoor = str(raw.get("outdoor")).lower() == "true"
    in_city = str(raw.get("likelyInCity", "")).lower()
    est.update(
        status="estimated",
        likelyInCity=in_city if in_city in LIKELY_IN_CITY else "unsure",
        eventKind=str(raw.get("eventKind") or "other"),
        sessions=sessions,
        outdoor=outdoor,
        capacityUsed=capacity,
        capacitySource=origin,
        radiusMeters=radius_from_attendance(est["expectedAttendance"], outdoor),
    )
    return est


def rule_v3(event, city=TARGET_CITY):
    ruled = rule_v2(event, city)
    if ruled:
        ruled.update(sessions=None, outdoor=None, radiusMeters=0 if ruled["expectedAttendance"] == 0 else None)
    return ruled


def estimate_attendance_v3(events, model=MODEL_V3, verbose=True):
    return estimate_attendance(
        events, model, verbose=verbose, version=PROMPT_VERSION_V3, system_prompt=SYSTEM_PROMPT_V3,
        input_fn=to_llm_input_v2, user_prompt_fn=build_user_prompt_v3, clean_fn=clean_estimate_v3, rule_fn=rule_v3,
    )

## 20. Run v3 (no web search) on the labelled events

In [ ]:
est_v3_lab = estimate_attendance_v3(label_events)
lab["v3"] = [(e or {}).get("expectedAttendance") for e in est_v3_lab]
lab["v3_radius"] = [(e or {}).get("radiusMeters") for e in est_v3_lab]
lab["v3_kind"] = [(e or {}).get("eventKind") for e in est_v3_lab]
lab["v3_reason"] = [((e or {}).get("reason") or "")[:80] for e in est_v3_lab]
print("usage this session:", USAGE.get(MODEL_V3))
score_columns(lab, ["v2", "v3"])

In [ ]:
biggest_misses(lab, "v3", extra=["v2", "v3_kind", "v3_reason"])

## 21. v3 + category priors enforced in code

v3 often ignores the priors in its prompt (it puts 4x4 leisure volleyball at 45-60, calls a free Saint-Eustache organ recital "invited audience, 40").
So here the model also names a **`priorCategory`** from a fixed list, and **code** clamps the attendance into that category's range.
Categories whose turnout follows capacity (concerts, club nights, pro sport in its own arena...) use `"none"` and are not clamped.
The ranges are the labelling rubric's category priors.

Two more code-side fixes in this version:
- **Known venue capacities**: a small table of well-known Paris venues (public figures) used as a verified ceiling. The model sometimes guesses a big arena's size badly (e.g. Adidas Arena ~8,000 guessed as 28,000).
- **Retry pass**: events a batch answer left out are asked again once (gpt-oss-20b often skips a few per batch).

In [ ]:
PROMPT_VERSION_V3P = "v3p.2"  # bump after editing the v3p prompt or venue table

# Well-known Paris venues -> max capacity (standing/main configuration). Used as a verified ceiling.
# Only venues with one main hall; multi-hall venues (e.g. Philharmonie) are left to the model.
KNOWN_VENUES = {
    "stade de france": 80000, "parc des princes": 48000, "la defense arena": 40000, "accor arena": 20300,
    "adidas arena": 8000, "zenith paris": 6300, "zenith de paris": 6300, "olympia": 2000, "salle pleyel": 1900,
    "casino de paris": 1500, "bataclan": 1500, "elysee montmartre": 1380, "la cigale": 1400, "trianon": 1000,
    "cabaret sauvage": 1200, "new morning": 400, "point ephemere": 300, "la maroquinerie": 500,
}


def known_venue_capacity(event):
    name = _ascii_upper(str(event.get("venue_name") or "")).lower()
    return next((cap for venue, cap in KNOWN_VENUES.items() if venue in name), None)


def verified_capacity_v3p(event):
    capacity, origin = verified_capacity(event)
    if capacity:
        return capacity, origin
    known = known_venue_capacity(event)
    return (known, "known venue table") if known else (None, None)


def to_llm_input_v3p(event):
    data = to_llm_input(event)
    capacity, origin = verified_capacity_v3p(event)
    if capacity:
        data["venue_capacity"] = capacity
        data["venue_capacity_source"] = origin
    return data


# priorCategory -> (min, max) people on the day; from the labelling rubric
CATEGORY_PRIORS = {
    "leisure_league": (10, 16),        # corporate/leisure leagues, e.g. FSGT 4x4 volleyball: players only
    "youth_sport": (25, 80),           # U13-U17: players and parents, whatever the hall size
    "amateur_regional": (30, 80),      # departmental/regional senior matches and cups
    "national_amateur": (50, 150),     # 4th-5th tier club sport, e.g. Nationale 2/3
    "ice_hockey_d2": (150, 400),       # ice hockey 3rd tier (Division 2)
    "pro_second_tier": (400, 1500),    # Ligue B, Pro B, Proligue
    "para_sport": (20, 40),            # para-sport lower divisions
    "talk_lecture": (40, 80),          # book talks, museum/library lectures, academic half-day seminars
    "workshop_networking": (15, 60),   # workshops, networking, afterworks
    "major_church_organ": (200, 500),  # free organ recitals in major churches
    "quiet_free_exhibition": (50, 150),
    "parish_youth": (20, 50),
    "religious_retreat": (8, 20),
    "community_fair": (300, 1000),     # municipal gaming or community fairs, per day
    "network_day": (150, 400),         # professional network days
}
PER_SESSION_PRIORS = {"guided_tour": (15, 30)}      # x sessions
CAPACITY_SHARE_PRIORS = {"jam_bar_night": (0.4, 0.6)}  # x capacity (free weekly jams / bar nights)
RSVP_SHARE_PRIORS = {"free_meetup": (0.75, 0.9)}      # x rsvp_count
PRIOR_KEYS = [*CATEGORY_PRIORS, *PER_SESSION_PRIORS, *CAPACITY_SHARE_PRIORS, *RSVP_SHARE_PRIORS, "none"]

SYSTEM_PROMPT_V3P = SYSTEM_PROMPT_V3.replace(
    '- "reason":',
    '- "priorCategory": the category prior that fits this event, exactly one of: ' + ", ".join(PRIOR_KEYS) + '. '
    'Use "none" when turnout follows the venue capacity (concerts, club nights, theatre, professional sport in its arena, '
    'big festivals) or when no prior fits. leisure_league = corporate/leisure leagues (FSGT, "Division 5-7", 4x4); '
    'youth_sport = U11-U18; amateur_regional = departmental/regional senior leagues and cups; '
    'national_amateur = Nationale 1-3 and similar 4th-5th tiers; para_sport = any wheelchair/handisport/para match '
    '(takes priority over the league level); talk_lecture = talks, lectures, seminars, conferences in a museum, library '
    'or university; workshop_networking = workshops, classes, afterworks, small networking evenings; network_day = a '
    'full-day national/professional federation or trade-network gathering; jam_bar_night = free or cheap weekly jam '
    'sessions and regular bar nights; guided_tour = guided visits and walking tours; free_meetup = free events with an '
    'rsvp_count.\n- "reason":',
).replace('"eventKind": "amateur_sport", "reason"', '"eventKind": "amateur_sport", "priorCategory": "amateur_regional", "reason"')


def prior_range(category, est, event):
    # (min, max) people for a priorCategory, or None (no clamp).
    if category in CATEGORY_PRIORS:
        return CATEGORY_PRIORS[category]
    if category in PER_SESSION_PRIORS:
        lo, hi = PER_SESSION_PRIORS[category]
        return lo * est["sessions"], hi * est["sessions"]
    if category in CAPACITY_SHARE_PRIORS and est.get("capacityUsed"):
        lo, hi = CAPACITY_SHARE_PRIORS[category]
        return max(1, round(lo * est["capacityUsed"])), max(1, round(hi * est["capacityUsed"]))
    rsvps = _number(event.get("attendees"))
    if category in RSVP_SHARE_PRIORS and rsvps:
        lo, hi = RSVP_SHARE_PRIORS[category]
        return max(1, round(lo * rsvps)), max(1, round(hi * rsvps))
    return None


def clean_estimate_v3p(raw, event):
    est = clean_estimate_v3(raw, event, capacity_fn=verified_capacity_v3p)
    if est is None:
        return None
    category = str(raw.get("priorCategory") or "none").strip().lower()
    category = category if category in PRIOR_KEYS else "none"
    bounds = prior_range(category, est, event)
    model_value = est["expectedAttendance"]
    if bounds:
        lo, hi = bounds
        if est.get("capacityUsed"):  # a real ceiling still wins over the prior
            hi = min(hi, est["capacityUsed"] * est["sessions"])
            lo = min(lo, hi)
        est["expectedAttendance"] = round_attendance(min(max(model_value, lo), hi))
        est["low"] = round_attendance(min(max(est["low"], lo), est["expectedAttendance"]))
        est["high"] = round_attendance(max(min(est["high"], hi), est["expectedAttendance"]))
        est["radiusMeters"] = radius_from_attendance(est["expectedAttendance"], est["outdoor"])
    est.update(priorCategory=category, priorRange=list(bounds) if bounds else None, modelAttendance=model_value)
    return est


def estimate_attendance_v3p(events, model=MODEL_V3, verbose=True, retries=1):
    run = lambda: estimate_attendance(
        events, model, verbose=verbose, version=PROMPT_VERSION_V3P, system_prompt=SYSTEM_PROMPT_V3P,
        input_fn=to_llm_input_v3p, user_prompt_fn=build_user_prompt_v3, clean_fn=clean_estimate_v3p, rule_fn=rule_v3,
    )
    results = run()
    for _ in range(retries):  # events the model left out of a batch answer are asked again (cached ones cost nothing)
        if all(r is not None for r in results):
            break
        results = run()
    return results


est_v3p_lab = estimate_attendance_v3p(label_events)
lab["v3p"] = [(e or {}).get("expectedAttendance") for e in est_v3p_lab]
lab["v3p_radius"] = [(e or {}).get("radiusMeters") for e in est_v3p_lab]
lab["prior"] = [(e or {}).get("priorCategory") for e in est_v3p_lab]
lab["model_said"] = [(e or {}).get("modelAttendance") for e in est_v3p_lab]
print("usage this session:", USAGE.get(MODEL_V3))
score_columns(lab, ["v2", "v3", "v3p"])

In [ ]:
# Which prior each event got, and where the clamp changed the model's number
lab[["id", "event", "label", "v3", "model_said", "v3p", "prior"]]

In [ ]:
biggest_misses(lab, "v3p", extra=["v3", "prior", "v3_reason"])

## 22. v3 + priors + web search for the biggest events

Only **concerts, festivals, shows and pro sport with a capacity of 3,000 or more** get a second, single-event request with Groq's `browser_search` tool (biggest first). The model searches for sell-out notices, the room's capacity and the act's previous Paris venue, then answers in the same JSON shape plus `evidence` and `sources`. Everything else keeps its v3 + priors answer.

**Cost:** one search request is **~60k tokens**, and the free tier allows **200k tokens per day** for gpt-oss-120b - so about 3 searches a day. `WEB_MAX_REQUESTS_PER_RUN` limits new searches per run, and a daily-limit error stops the searching cleanly (those events keep their non-search answer; re-run later - answers already found are cached under `version = "v3-web"`).

In [ ]:
WEB_VERSION = "v3-web"
WEB_MIN_CAPACITY = 3000
WEB_KINDS = {"concert", "festival", "theatre_show", "club_night", "pro_sport"}
WEB_MAX_REQUESTS_PER_RUN = 3  # free tier: ~60k tokens each, 200k tokens/day


WEB_ADDENDUM = """

WEB SEARCH
You have browser search. Use it for this event (at least 3 searches) to find, in this order:
1. What the event really is (open its url if one is given).
2. The capacity of THIS configuration (standing or seated, which room or hall).
3. Demand signals: "sold out", "complet" or "indisponible" on official ticketing (Ticketmaster, Fnac Spectacles, DICE, AXS, Shotgun, See Tickets); extra dates added for demand; only resale left; the act's previous {CITY} venue and whether it sold out.
4. Other big events at or next to the venue the same day.
Only use sell-outs, counts and caps that a page you actually found states. If you find nothing, say "no evidence found" and use the rubric.
Add two fields to the estimate: "evidence" (max 30 words: what you found) and "sources" (list of website domains).
Return ONLY the JSON object, no other text.""".replace("{CITY}", TARGET_CITY)


def _request_with_search(model, messages, max_attempts=6):
    for attempt in range(max_attempts):
        try:
            return client.chat.completions.create(
                model=model, messages=messages, temperature=0, reasoning_effort="medium",
                tools=[{"type": "browser_search"}], tool_choice="required",
            )
        except groq.RateLimitError as error:
            if "tokens per day" in str(error):
                raise DailyTokenLimit(str(error)) from None
            wait = min(90.0, float(error.response.headers.get("retry-after") or 5 * 2**attempt))
            print(f"  rate limited - waiting {wait:.0f}s")
            time.sleep(wait)
        except (groq.APIConnectionError, groq.InternalServerError) as error:
            print(f"  {type(error).__name__} - retrying")
            time.sleep(2**attempt)
    raise RuntimeError(f"{model}: no usable search response after {max_attempts} attempts")


def _json_from_text(text):
    """First {...} object with an "estimates" list in free text (search answers can't use JSON mode)."""
    text = text or ""
    for match in re.finditer(r"\{", text):
        try:
            obj, _ = json.JSONDecoder().raw_decode(text[match.start():])
        except ValueError:
            continue
        if isinstance(obj, dict) and isinstance(obj.get("estimates"), list):
            return obj
    return {}


def needs_search(est):
    return (
        bool(est) and est.get("status") == "estimated" and est.get("eventKind") in WEB_KINDS
        and (est.get("capacityUsed") or 0) >= WEB_MIN_CAPACITY
    )


def estimate_attendance_web(events, base_estimates, model=MODEL_V3, verbose=True):
    """Re-estimate the biggest events with web search; everything else keeps its base answer."""
    system_prompt = SYSTEM_PROMPT_V3P + WEB_ADDENDUM
    usage = USAGE.setdefault(
        model + " + browser_search", {"requests": 0, "prompt_tokens": 0, "completion_tokens": 0, "seconds": 0.0}
    )
    results = list(base_estimates)
    picked = sorted(
        (i for i, est in enumerate(base_estimates) if needs_search(est)),
        key=lambda i: -base_estimates[i]["capacityUsed"],
    )
    if verbose:
        print(f"{len(picked)} of {len(events)} events qualify for web search (capacity >= {WEB_MIN_CAPACITY})")
    new_requests, stopped = 0, None
    for i in picked:
        data = to_llm_input_v3p(events[i])
        key = cache_key(model, data, WEB_VERSION)
        if key not in CACHE and not stopped and new_requests >= WEB_MAX_REQUESTS_PER_RUN:
            stopped = f"WEB_MAX_REQUESTS_PER_RUN = {WEB_MAX_REQUESTS_PER_RUN} reached"
        if key not in CACHE and not stopped:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": build_user_prompt_v3([(1, data)])},
            ]
            started = time.time()
            try:
                response = _request_with_search(model, messages)
            except DailyTokenLimit as error:
                stopped = "daily token limit reached (" + str(error).split("Please try again in")[-1].split(".")[0].strip() + ")"
                continue
            new_requests += 1
            usage["requests"] += 1
            usage["seconds"] += time.time() - started
            if response.usage:
                usage["prompt_tokens"] += response.usage.prompt_tokens
                usage["completion_tokens"] += response.usage.completion_tokens
            items = _json_from_text(response.choices[0].message.content).get("estimates") or []
            raw = items[0] if items and isinstance(items[0], dict) else None
            cleaned = clean_estimate_v3p(raw, events[i]) if raw else None
            if cleaned:
                sources = raw.get("sources") if isinstance(raw.get("sources"), list) else []
                cleaned.update(evidence=str(raw.get("evidence") or "")[:300], sources=sources, searched=True)
                CACHE[key] = cleaned
                save_cache()
            if verbose:
                name = (events[i].get("event_name") or "")[:40]
                print(f"  {name}: {'ok' if cleaned else 'no usable answer - keeping v3'} in {time.time() - started:.0f}s")
        if key in CACHE:
            results[i] = CACHE[key]
    if verbose:
        done = sum(1 for i in picked if (results[i] or {}).get("searched"))
        print(f"{done}/{len(picked)} have a web answer" + (f" - stopped: {stopped}; re-run later for the rest" if stopped else ""))
    return results


est_web_lab = estimate_attendance_web(label_events, est_v3p_lab)
lab["v3_web"] = [(e or {}).get("expectedAttendance") for e in est_web_lab]
lab["v3_web_radius"] = [(e or {}).get("radiusMeters") for e in est_web_lab]
lab["searched"] = [bool((e or {}).get("searched")) for e in est_web_lab]
lab["evidence"] = [((e or {}).get("evidence") or "")[:90] for e in est_web_lab]
lab.loc[lab["searched"], ["id", "event", "label", "v3p", "v3_web", "evidence"]]

## 23. Side by side

Attendance and radius, all scored rows, then split into big (label ≥ 500) and small events - web search only touches the big ones.
v2 has no radius of its own, so its radius column uses the same tier function on v2's attendance.

In [ ]:
lab["v2_radius"] = lab["v2"].map(radius_from_attendance)
big = lab["label"] >= 500
pd.concat(
    {
        "attendance - all": score_columns(lab, ["v2", "v3", "v3p", "v3_web"]),
        "attendance - big (label >= 500)": score_columns(lab, ["v2", "v3", "v3p", "v3_web"], mask=big),
        "attendance - small": score_columns(lab, ["v2", "v3", "v3p", "v3_web"], mask=~big),
        "radius - all": score_columns(lab, ["v2_radius", "v3_radius", "v3p_radius", "v3_web_radius"], truth_col="label_radius")
        .set_axis(["v2", "v3", "v3p", "v3_web"], axis=1),
    }
)

In [ ]:
# Requests, tokens and time used in this session (cached answers cost nothing)
pd.DataFrame(USAGE).T

In [ ]:
# Biggest remaining misses of the best version
biggest_misses(lab, "v3_web", n=12, extra=["v3", "v3p", "prior", "v3_reason"])

In [ ]:
# Everything in one table (held-out example rows marked scored=False)
lab[["id", "event", "scored", "label", "v2", "v3", "v3p", "v3_web", "label_radius", "v3p_radius", "v3_web_radius", "prior"]]

## 24. Venue address for sports events with no location

657 upcoming sportsevents24 fixtures arrive with an empty `location`, so they can't be geocoded and
are dropped. The home team is the clue: the model names the **venue + postal address** where the
home side plays (it never outputs coordinates). The address then goes through LocationIQ, and the
Paris boundary check says whether the fixture is in the city.

Same prompt, same random sample, three models -> `notebooks/venue_location_check.csv` for manual checking.

In [ ]:
import importlib, sys as _sys
if str(PROJECT_ROOT) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_ROOT))
import core.geocoding as geocoding
from connectors.sports.sportsevents24.connector import SPORTS, load_boundary, in_boundary

RAW_DB = PROJECT_ROOT / "core" / "raw_events.db"
LOC_VERSION = "loc-v1"          # bump after editing the prompt -> cached answers are ignored
LOC_MODELS = ["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b"]
LOC_BATCH_SIZES = {"openai/gpt-oss-20b": 6, "openai/gpt-oss-120b": 6, "qwen/qwen3.8-27b": 2}
LOC_SAMPLE_SEED = 7
LOC_CSV = PROJECT_ROOT / "notebooks" / "venue_location_check.csv"
GEO_CACHE_PATH = PROJECT_ROOT / "notebooks" / "venue_guess_geocode_cache.json"
PARIS_POLYGONS = load_boundary("Paris")


def load_locationless_fixtures(from_date):
    """Distinct sportsevents24 fixtures with an empty location, from `from_date` on."""
    con = sqlite3.connect(f"file:{RAW_DB}?mode=ro", uri=True)
    rows = con.execute("SELECT raw_json FROM raw_events WHERE source = 'sportsevents24' ORDER BY id DESC").fetchall()
    con.close()
    seen, out = set(), []
    for (blob,) in rows:
        r = json.loads(blob)
        if r["id"] in seen or (r.get("location") or "").strip() or r["localStartsAt"][:10] < from_date:
            continue
        seen.add(r["id"])
        out.append(r)
    return out


fixtures = load_locationless_fixtures(str(date.today()))
print(len(fixtures), "upcoming fixtures without a location")

In [ ]:
PRO_RE = re.compile(r"ligue 1|ligue 2|starligue|proligue|pro d2|top 14|betclic|lnh|marmara|ligue [ab]\b|\bnationale?\s*[123]?\b|elite|élite|reichel", re.I)
NOT_PRO_RE = re.compile(r"pr[ée][ -]?nationa|\b[mu]\s?1[0-8]\b|régional|regional", re.I)
IDF_RE = re.compile(r"ile-de-france|île-de-france|\bparis|parisien|red star|stade fran|racing 92|creteil|créteil|acbb", re.I)


def pick_location_sample(fixtures, per_group=8, seed=LOC_SAMPLE_SEED):
    """Random but spread: Paris-region teams, pro/national leagues, and amateur anywhere.
    One fixture per home team so the comparison isn't repeated."""
    rng = random.Random(seed)
    pool = list(fixtures)
    rng.shuffle(pool)
    groups = {"paris_region": [], "pro_national": [], "amateur": []}
    used_teams = set()
    for r in pool:
        team = r["teamHome"].strip().lower()
        if team in used_teams:
            continue
        text = f'{r["tournament"]} {r["teamHome"]}'
        group = ("paris_region" if IDF_RE.search(text)
                 else "pro_national" if PRO_RE.search(r["tournament"]) and not NOT_PRO_RE.search(r["tournament"])
                 else "amateur")
        if len(groups[group]) < per_group:
            groups[group].append({**r, "group": group})
            used_teams.add(team)
    return [r for g in groups.values() for r in g]


loc_sample = pick_location_sample(fixtures)
pd.DataFrame([{"group": r["group"], "date": r["localStartsAt"][:10], "sport": SPORTS.get(r["sportId"]),
               "tournament": r["tournament"][:60], "home": r["teamHome"], "away": r["teamAway"]}
              for r in loc_sample])

In [ ]:
SYSTEM_PROMPT_LOC = """You are an expert on sports clubs in France and Europe and on the venues where they play their home matches.
For each fixture, say where the match is played and give that venue's postal address.

HOW TO WORK IT OUT
- The first team ("home") hosts the match at its own home venue, unless the competition name says it is a
  tournament / plateau / brassage / tournoi / finals day / neutral venue (then say so and lower confidence).
- Find the home club: strip team numbers and suffixes ("2", "3", "B", "U19", "M18", "Espoirs", "Féminines", "Rst ..."),
  expand abbreviations: AS, US, FC, SC, ES, CS, RC, AC, ASPTT, SCUF, "ENT." / "Entente" = merger of clubs,
  "IE - CTC" = cooperation of several clubs (the home club is the first one named; its town is the venue town).
  "Rst Scuf/red Star M19" = a joint youth side of SCUF and Red Star.
- Professional and semi-pro first teams play at their known main stadium/arena (e.g. Paris Saint-Germain -> Parc des Princes,
  Paris Handball -> Stade Pierre de Coubertin, Red Star FC -> Stade Bauer in Saint-Ouen, not Paris).
- Reserve, youth and women's sides often do NOT play at the main stadium: they use the club's training centre or a
  smaller municipal ground. Use the training ground when you know it; otherwise say the venue is uncertain.
- Amateur clubs play at a municipal stadium (football, rugby) or gymnase / salle / palais des sports (basketball,
  volleyball, handball, futsal) in their own commune. The commune is usually in the club name
  ("PANTIN VOLLEY" -> Pantin, "AS AVION FUTSAL" -> Avion, "ST AMAND FC" -> Saint-Amand-les-Eaux if the league region fits).
  Use the league / region in the competition name to pick between towns with the same name.
- A club name containing "Paris" is not proof the venue is inside the city of Paris (e.g. Saint-Cloud Paris SF plays in Saint-Cloud).

HONESTY RULES - a wrong address is worse than a missing one
- Give venueName and streetAddress only if you actually know them. Never invent a street name or street number.
- If you know the town but not the exact venue: venueName null, streetAddress null, town + postcode filled in.
- If you cannot tell the town either: everything null and confidence "low".
- Never output coordinates.
- confidence: "high" = you know this club plays this level of match at this venue; "medium" = it is the club's usual venue
  but this team/level might play elsewhere, or part of the address is uncertain; "low" = an educated guess.

OUTPUT: one JSON object {"results": [ ... ]} with exactly one entry per input id:
{"id": <same id>, "homeClub": "<club name, expanded>", "venueName": str|null, "streetAddress": "<number + street>"|null,
 "postcode": str|null, "town": str|null, "country": "<country name>", "venueType": "stadium"|"gymnasium"|"arena"|
 "training_ground"|"other"|"unknown", "confidence": "high"|"medium"|"low", "reason": "<= 25 words: how you know"}"""


def to_loc_input(r):
    return {"sport": SPORTS.get(r["sportId"], str(r["sportId"])), "competition": r["tournament"],
            "home": r["teamHome"], "away": r["teamAway"], "date": r["localStartsAt"][:10], "country": r["country"]}


def build_user_prompt_loc(batch):
    lines = [f"Fixtures ({len(batch)}). Return one result per id."]
    for i, inp in batch:
        lines.append(json.dumps({"id": i, **inp}, ensure_ascii=False))
    return "\n".join(lines)


def _clean_str(value):
    value = str(value).strip() if value is not None else ""
    return None if value.lower() in {"", "null", "none", "unknown", "n/a"} else value


def clean_location(raw):
    out = {k: _clean_str(raw.get(k)) for k in ["homeClub", "venueName", "streetAddress", "postcode", "town", "country", "venueType", "reason"]}
    conf = str(raw.get("confidence", "")).lower()
    out["confidence"] = conf if conf in CONFIDENCE_LEVELS else "low"
    return out


def locate_venues(fixtures, model, verbose=True):
    """Location dicts (or None) aligned with `fixtures`; cached per (version, model, input)."""
    results = [None] * len(fixtures)
    todo = []
    for n, r in enumerate(fixtures):
        inp = to_loc_input(r)
        key = cache_key(model, inp, version=LOC_VERSION)
        if key in CACHE:
            results[n] = CACHE[key]
        else:
            todo.append((n, inp, key))
    def ask(chunk):
        """{local id: raw item} for one request; raises on an unusable reply."""
        messages = [{"role": "system", "content": SYSTEM_PROMPT_LOC},
                    {"role": "user", "content": build_user_prompt_loc([(k + 1, inp) for k, (_, inp, _) in enumerate(chunk)])}]
        t0 = time.time()
        response = _request(model, messages, max_attempts=3)
        u = USAGE.setdefault(model, {"requests": 0, "tokens": 0, "seconds": 0.0})
        u["requests"] += 1; u["tokens"] += response.usage.total_tokens; u["seconds"] += time.time() - t0
        items = json.loads(response.choices[0].message.content).get("results", [])
        return {int(it["id"]): it for it in items if isinstance(it, dict) and str(it.get("id", "")).isdigit()}

    def store(chunk, by_id):
        for k, (n, inp, key) in enumerate(chunk):
            if k + 1 in by_id:
                results[n] = CACHE[key] = clean_location(by_id[k + 1])

    size = LOC_BATCH_SIZES.get(model, 6)
    for start in range(0, len(todo), size):
        chunk = todo[start:start + size]
        try:
            try:
                store(chunk, ask(chunk))
            except (groq.BadRequestError, RuntimeError, json.JSONDecodeError, AttributeError) as error:
                # gpt-oss-20b sometimes fails a whole batch on invalid JSON (deterministic at temperature 0) ->
                # ask one fixture at a time so only the broken one is lost.
                print(f"  batch failed ({type(error).__name__}) - retrying one fixture at a time")
                for single in ([c] for c in chunk):
                    try:
                        store(single, ask(single))
                    except (groq.BadRequestError, RuntimeError, json.JSONDecodeError, AttributeError) as e2:
                        print(f"    skipped one fixture ({type(e2).__name__})")
        except DailyTokenLimit as error:
            print("STOPPED -", error, f"| {len(todo) - start} fixtures left for {model}")
            break
        save_cache()
        if verbose:
            print(f"  {model}: {min(start + size, len(todo))}/{len(todo)} done")
    return results

In [ ]:
GEO_CACHE = json.loads(GEO_CACHE_PATH.read_text(encoding="utf-8")) if GEO_CACHE_PATH.exists() else {}


def address_text(loc):
    """Full address for the geocoder (venue name dropped when a street is known - it confuses LocationIQ)."""
    if not loc:
        return None
    parts = [loc.get("streetAddress") or loc.get("venueName"), " ".join(p for p in [loc.get("postcode"), loc.get("town")] if p), loc.get("country")]
    parts = [p for p in parts if p]
    return ", ".join(parts) if (loc.get("town") or loc.get("postcode")) else None


def geocode_guess(loc):
    """LocationIQ lookup of the model's address, cached in a notebook JSON file
    (deliberately NOT in core/locations.db - these are unverified guesses)."""
    query = address_text(loc)
    if not query:
        return {"query": None, "lat": None, "lon": None, "geoTown": None, "inParis": None}
    if query not in GEO_CACHE:
        hit = geocoding._geocode(query)
        a = (hit or {}).get("address", {})
        GEO_CACHE[query] = None if not hit else {
            "lat": float(hit["lat"]), "lon": float(hit["lon"]),
            "geoTown": a.get("city") or a.get("town") or a.get("village") or a.get("municipality"),
            "type": hit.get("type")}
        GEO_CACHE_PATH.write_text(json.dumps(GEO_CACHE, ensure_ascii=False, indent=1), encoding="utf-8")
    g = GEO_CACHE[query] or {}
    lat, lon = g.get("lat"), g.get("lon")
    return {"query": query, "lat": lat, "lon": lon, "geoTown": g.get("geoTown"),
            "inParis": None if lat is None else in_boundary(lat, lon, PARIS_POLYGONS)}

In [ ]:
LOC_RESULTS = {}
for m in LOC_MODELS:
    print("==", m)
    LOC_RESULTS[m] = locate_venues(loc_sample, m)
    print("   answered", sum(x is not None for x in LOC_RESULTS[m]), "/", len(loc_sample), "| usage:", USAGE.get(m))

In [ ]:
SHORT = {"openai/gpt-oss-20b": "20b", "openai/gpt-oss-120b": "120b", "qwen/qwen3.8-27b": "qwen"}


def maps_link(lat, lon):
    return f"https://www.google.com/maps?q={lat:.6f},{lon:.6f}" if lat is not None else None


rows = []
for n, r in enumerate(loc_sample, start=1):
    row = {"#": n, "group": r["group"], "date": r["localStartsAt"][:10], "sport": SPORTS.get(r["sportId"]),
           "competition": r["tournament"], "home": r["teamHome"], "away": r["teamAway"]}
    for m in LOC_MODELS:
        s = SHORT[m]
        loc = LOC_RESULTS[m][n - 1]
        g = geocode_guess(loc)
        row[f"{s}_venue"] = (loc or {}).get("venueName")
        row[f"{s}_address"] = ", ".join(p for p in [(loc or {}).get("streetAddress"), (loc or {}).get("postcode"), (loc or {}).get("town")] if p) or None
        row[f"{s}_confidence"] = (loc or {}).get("confidence")
        row[f"{s}_in_paris"] = g["inParis"]
        row[f"{s}_map"] = maps_link(g["lat"], g["lon"])
        row[f"{s}_reason"] = (loc or {}).get("reason")
    for s in SHORT.values():  # for you to fill: correct / town_only / wrong
        row[f"check_{s}"] = ""
    row["true_venue_address"] = ""
    rows.append(row)

loc_table = pd.DataFrame(rows)
loc_table.to_csv(LOC_CSV, index=False)
print("written", LOC_CSV, "| geocoder calls cached:", len(GEO_CACHE))
loc_table[["#", "group", "home", "away"] + [f"{s}_{f}" for s in SHORT.values() for f in ("venue", "address", "confidence", "in_paris")]]